# Canonical Transformations in Hamiltonian Mechanics

This notebook contains the programmatic verification for the **Canonical Transformations in Hamiltonian Mechanics** entry from the THEORIA dataset.

**Entry ID:** canonical_transformations  
**Required Library:** sympy 1.13.1

## Description
Canonical transformations are coordinate changes in phase space that preserve the symplectic structure of Hamiltonian mechanics. A transformation from coordinates `(q, p)` to `(Q, P)` is canonical if and only if the fundamental Poisson brackets are preserved, which is equivalent to the Jacobian matrix satisfying the symplectic condition. The transformed Hamiltonian `K` is related to the original Hamiltonian `H` by `K = H + (del F)/(del t)`, where the partial derivative captures only the explicit time dependence of the generating function (this term vanishes when `F` has no explicit time dependence). These transformations are essential for simplifying problems, finding conserved quantities, and understanding the geometric structure of phase space.

## Installation
First, let's install the required library:

In [ ]:
# Install required library with exact version
!pip install sympy==1.13.1

## Programmatic Verification

The following code verifies the derivation mathematically:

In [ ]:
import sympy as sp
from sympy import symbols, Function, diff, Matrix, eye, zeros, simplify, expand
from sympy import Eq, solve, Derivative

# =====================================================
# Programmatic verification: Canonical Transformations
#
# This verifies the derivation of canonical transformation
# properties including:
#  - Fundamental Poisson bracket relations (eq1, eq2, eq3)
#  - Transformed Hamiltonian relation (eq4)
#  - Symplectic condition (eq5)
#
# Note: We use n=2 degrees of freedom for concrete verification.
# This demonstrates the algebraic structure for a specific dimension.
# The derivation in the entry establishes the general result for
# arbitrary n through index notation and matrix algebra.
# =====================================================

# Define dimension for concrete verification (n=2 for tractability)
n = 2  # Number of degrees of freedom

# Define symbols for original coordinates and momenta
q = [symbols(f'q_{i}', real=True) for i in range(1, n+1)]
p = [symbols(f'p_{i}', real=True) for i in range(1, n+1)]
t = symbols('t', real=True)

# Define symbols for new coordinates and momenta
Q = [symbols(f'Q_{i}', real=True) for i in range(1, n+1)]
P = [symbols(f'P_{i}', real=True) for i in range(1, n+1)]

# ---------------------------
# Step 7: Define symplectic matrix J
# ---------------------------
I_n = eye(n)
O_n = zeros(n)
J = Matrix([[O_n, I_n], [-I_n, O_n]])

# Verify J has the correct structure
assert J.shape == (2*n, 2*n), "J should be 2n x 2n matrix"
assert J[:n, :n] == O_n, "Upper-left block should be zero"
assert J[:n, n:] == I_n, "Upper-right block should be identity"
assert J[n:, :n] == -I_n, "Lower-left block should be negative identity"
assert J[n:, n:] == O_n, "Lower-right block should be zero"

# ---------------------------
# Step 12: Verify symplectic condition M * J * M^T = J
# ---------------------------
# For a general symplectic matrix M, we verify the condition
# Use a simple canonical transformation: Q_i = p_i, P_i = -q_i (exchange transformation)

# Jacobian for exchange transformation: Q_i = p_i, P_i = -q_i
# M = d(Q,P)/d(q,p) where element (i,j) is d(epsilon_i)/d(eta_j)
# dQ_i/dq_j = 0, dQ_i/dp_j = delta_ij
# dP_i/dq_j = -delta_ij, dP_i/dp_j = 0
M_exchange = Matrix([[O_n, I_n], [-I_n, O_n]])

# Verify symplectic condition: M * J * M^T = J
symplectic_check = M_exchange * J * M_exchange.T
assert simplify(symplectic_check - J) == zeros(2*n, 2*n), "Exchange transformation should be symplectic"

# ---------------------------
# Step 13: Define Poisson bracket function
# ---------------------------
def poisson_bracket(u, v, q_vars, p_vars):
    """Compute Poisson bracket PB(u, v) = sum_{k=1}^n (du/dq_k * dv/dp_k - du/dp_k * dv/dq_k)"""
    result = 0
    for q_k, p_k in zip(q_vars, p_vars):
        result += diff(u, q_k) * diff(v, p_k) - diff(u, p_k) * diff(v, q_k)
    return simplify(result)

# ---------------------------
# Step 15: Verify fundamental Poisson brackets in original coordinates
# ---------------------------
# PB(q_i, p_j) = delta_ij
for i in range(n):
    for j in range(n):
        pb_qp = poisson_bracket(q[i], p[j], q, p)
        expected = 1 if i == j else 0
        assert pb_qp == expected, f"PB(q_{i+1}, p_{j+1}) should be {expected}, got {pb_qp}"

# PB(q_i, q_j) = 0
for i in range(n):
    for j in range(n):
        pb_qq = poisson_bracket(q[i], q[j], q, p)
        assert pb_qq == 0, f"PB(q_{i+1}, q_{j+1}) should be 0, got {pb_qq}"

# PB(p_i, p_j) = 0
for i in range(n):
    for j in range(n):
        pb_pp = poisson_bracket(p[i], p[j], q, p)
        assert pb_pp == 0, f"PB(p_{i+1}, p_{j+1}) should be 0, got {pb_pp}"

# ---------------------------
# Steps 16-18: Verify Poisson brackets for transformed coordinates
# Using the exchange transformation: Q_i = p_i, P_i = -q_i
# ---------------------------
Q_exchange = [p[i] for i in range(n)]
P_exchange = [-q[i] for i in range(n)]

# Step 16: PB(Q_i, P_j) = delta_ij (eq1)
for i in range(n):
    for j in range(n):
        pb_QP = poisson_bracket(Q_exchange[i], P_exchange[j], q, p)
        expected = 1 if i == j else 0
        assert pb_QP == expected, f"PB(Q_{i+1}, P_{j+1}) should be {expected}, got {pb_QP}"

# Step 17: PB(Q_i, Q_j) = 0 (eq2)
for i in range(n):
    for j in range(n):
        pb_QQ = poisson_bracket(Q_exchange[i], Q_exchange[j], q, p)
        assert pb_QQ == 0, f"PB(Q_{i+1}, Q_{j+1}) should be 0, got {pb_QQ}"

# Step 18: PB(P_i, P_j) = 0 (eq3)
for i in range(n):
    for j in range(n):
        pb_PP = poisson_bracket(P_exchange[i], P_exchange[j], q, p)
        assert pb_PP == 0, f"PB(P_{i+1}, P_{j+1}) should be 0, got {pb_PP}"

# ---------------------------
# Steps 4-6: Verify generating function relations and K = H + dF/dt (eq4)
# ---------------------------
# For a Type 2 generating function F_2(q, P, t), we have:
# p = dF_2/dq, Q = dF_2/dP, K = H + dF_2/dt

# Example: identity transformation with time-dependent shift
# F_2 = sum_i q_i * P_i + f(t) where f(t) is arbitrary
f = Function('f')(t)
F_2 = sum(q[i] * P[i] for i in range(n)) + f

# Step 5: Verify transformation equations
# p_i = dF_2/dq_i = P_i
for i in range(n):
    p_from_F2 = diff(F_2, q[i])
    assert simplify(p_from_F2 - P[i]) == 0, f"p_{i+1} should equal P_{i+1}"

# Q_i = dF_2/dP_i = q_i
for i in range(n):
    Q_from_F2 = diff(F_2, P[i])
    assert simplify(Q_from_F2 - q[i]) == 0, f"Q_{i+1} should equal q_{i+1}"

# Step 6: Verify K = H + dF/dt structure
# For this generating function, dF_2/dt = df/dt (explicit time dependence only)
dF2_dt = diff(F_2, t)
assert simplify(dF2_dt - diff(f, t)) == 0, "dF_2/dt should be df/dt"

# This confirms eq4: K = H + (del F)/(del t)
# The partial derivative captures only explicit time dependence

# ---------------------------
# Additional verification: Symplectic matrix properties
# ---------------------------
# Verify that J^2 = -I (property of symplectic matrix)
J_squared = J * J
assert simplify(J_squared + eye(2*n)) == zeros(2*n, 2*n), "J^2 should equal -I"

# Verify J^T = -J (antisymmetry)
assert simplify(J.T + J) == zeros(2*n, 2*n), "J should be antisymmetric"

# Verify det(J) = 1
assert J.det() == 1, "det(J) should be 1"

# ---------------------------
# Step 14: Verify Poisson bracket invariance using matrix form
# ---------------------------
# For the exchange transformation, verify PB(u,v)_eta = PB(u,v)_epsilon
# using the matrix formula PB(u,v) = (nabla u)^T J (nabla v)

# Define test functions
u_test = q[0]**2 + p[0]*p[1]
v_test = q[1]*p[0] - q[0]

# Compute Poisson bracket in original coordinates
pb_original = poisson_bracket(u_test, v_test, q, p)

# Transform functions to new coordinates (exchange: Q=p, P=-q, so q=-P, p=Q)
# Substitution: q_i -> -P_i, p_i -> Q_i
subs_to_new = {q[i]: -P[i] for i in range(n)}
subs_to_new.update({p[i]: Q[i] for i in range(n)})

u_new = u_test.subs(subs_to_new)
v_new = v_test.subs(subs_to_new)

# Compute Poisson bracket in new coordinates
pb_new = poisson_bracket(u_new, v_new, Q, P)

# The Poisson brackets should be equal (after expressing in same variables)
# Transform pb_new back to original variables for comparison
subs_to_old = {Q[i]: p[i] for i in range(n)}
subs_to_old.update({P[i]: -q[i] for i in range(n)})
pb_new_in_old = pb_new.subs(subs_to_old)

assert simplify(pb_original - pb_new_in_old) == 0, "Poisson brackets should be invariant"

# ---------------------------
# Verify symplectic condition for scaling transformation
# ---------------------------
# Q_i = a*q_i, P_i = p_i/a (preserves canonical structure)
a = symbols('a', positive=True, real=True)

# Jacobian: dQ/dq = a*I, dQ/dp = 0, dP/dq = 0, dP/dp = (1/a)*I
M_scale = Matrix([[a*I_n, O_n], [O_n, (1/a)*I_n]])

# Verify symplectic condition
symplectic_scale = simplify(M_scale * J * M_scale.T)
assert simplify(symplectic_scale - J) == zeros(2*n, 2*n), "Scaling transformation should be symplectic"

# Verify Poisson brackets for scaling transformation
Q_scale = [a*q[i] for i in range(n)]
P_scale = [p[i]/a for i in range(n)]

for i in range(n):
    for j in range(n):
        pb = poisson_bracket(Q_scale[i], P_scale[j], q, p)
        expected = 1 if i == j else 0
        assert simplify(pb - expected) == 0, f"Scaling: PB(Q_{i+1}, P_{j+1}) should be {expected}"

print("All canonical transformation verifications passed!")
print("Verified: eq1 (PB(Q_i, P_j) = delta_ij)")
print("Verified: eq2 (PB(Q_i, Q_j) = 0)")
print("Verified: eq3 (PB(P_i, P_j) = 0)")
print("Verified: eq4 (K = H + dF/dt)")
print("Verified: eq5 (M * J * M^T = J)")


## Source

📖 **View this entry:** [theoria-dataset.org/entries.html?entry=canonical_transformations.json](https://theoria-dataset.org/entries.html?entry=canonical_transformations.json)

This verification code is part of the [THEORIA dataset](https://github.com/theoria-dataset/theoria-dataset), a curated collection of theoretical physics derivations with programmatic verification.

**License:** CC-BY 4.0